# Module 1 -- DRG-Net replication runs (classification + segmentation)

Re-runs DRG-Net's own training code (pinned commit `0da1bbe`) the way **they** used each dataset,
so our Module 1 numbers can be compared with theirs. DRG-Net trains **one model per dataset**
(never pooled), each with its own config:

| Task | Dataset | DRG-Net script + config | Split | Epochs |
|---|---|---|---|---|
| Grading | FGADR (1,842) | `main.py` + `fgadr.yaml` (SGD, weighted CE) | rows 0-1499 train, 1500-1841 test, **val = test** (their released `fgadr_data_dict.pkl`) | 20 |
| Grading | IDRiD (413 / 103) | `main.py` + `idrid.yaml` (Adam, focal loss, progressively balanced) | official train / test, **val = test** | 60 |
| Grading | EyePACS | `main.py` + `eyepacs_ce.yaml` (SGD, plain CE) | not released -- we use the Kaggle split (train / public test / private test) | 25 |
| Lesion seg. EX/HE/MA/SE | FGADR (401 with all 4 masks) | `train_fgadr.py` + `config_fgadr.py`, `--seed 765 --preprocess 2` | 60 / 20 / 20 by row order of `DR_Seg_Grading_Label_Filtered.csv` | 1500 (default here: **100**) |
| Lesion seg. EX/HE/MA/SE | IDRiD (54 / 27) | `train.py` + `config.py`, `--seed 765 --preprocess 2` | 80 / 20 of training set; test = official test set | 2500 (default here: **100**) |

EyePACS has **no lesion masks**, so DRG-Net never trains segmentation on it (their
`eyepacs_dataset.py` only loads images + FOV masks, unused by any training script).

**Known gaps vs the paper -- state these when comparing numbers:**
- The paper's headline segmentation results (e.g. FGADR "S-Net + PD-Net + TATL", the table in
  `module1/README.md`) use their TATL transfer-learning setup, which is **not in the released
  code**. These runs reproduce the released `NO_TATL` training, so expect lower numbers than the
  TATL row and compare against the paper's non-TATL baseline where it reports one.
- `val == test` for FGADR/IDRiD grading is DRG-Net's own protocol (checkpoint selection sees the
  test set). Kept for replication.
- **Epochs:** segmentation defaults to **100** (`SEG_EPOCHS`) instead of DRG-Net's 1500 / 2500
  -- report it as "DRG-Net's training protocol at a reduced epoch budget", not as a full
  replication. Their StepLR (x0.9) only every 200 epochs, so at 100 epochs the LR
  stays at 1e-3 throughout. Classification keeps DRG-Net's own 20 / 60 / 25 (`CLS_EPOCHS`
  overrides them).
- **Checkpoint / resume:** `module1/drgnet_resume_patch.py` adds resume to DRG-Net's training
  scripts (they can't resume on their own -- see that file). Checkpoints are written straight to
  Drive; if Colab disconnects, re-run the notebook and every run continues from its last saved
  epoch (classification: every epoch; segmentation: every `SEG_SAVE_EVERY` epochs). Tested:
  an interrupted + resumed classification run gives bit-identical weights to an uninterrupted
  one; segmentation resumes the epoch count, LR schedule and best-AP tracking (its data shuffling
  differs after a resume, so it is close but not bit-identical).
- Modern torch/torchvision/Pillow instead of their 2022 pins (same compatibility patches as
  notebooks 02/03), so expect small run-to-run variance.

Everything is written to `Thesis_Datasets/module1_runs/drgnet_replication/` on Drive, separate
from the POC runs of notebooks 02/03 that the rest of the pipeline uses.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os, glob, json
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'mod-2-NonLongIntegration'  # <-- change to 'main' once this branch is merged
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache')
MODULE2_RUNS_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs')
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
os.makedirs(MODULE2_RUNS_DIR, exist_ok=True)

REPL_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_runs', 'drgnet_replication')
REPL_PKL_DIR = os.path.join(REPL_DIR, 'pkl')
for _d in ('pkl', 'classification', 'segmentation', 'logs'):
    os.makedirs(os.path.join(REPL_DIR, _d), exist_ok=True)

# Which runs to do this session (each is long -- split them across sessions if needed).
RUN = {
    'fgadr_cls': True,
    'idrid_cls': True,
    'eyepacs_cls': False,   # needs the EyePACS zip (notebook 09) and several hours
    'fgadr_seg': True,
    'idrid_seg': True,      # needs IDRiD's "A. Segmentation" part in archive.zip
}
SEG_LESIONS = ['EX', 'HE', 'MA', 'SE']
SEG_EPOCHS = {'fgadr': 100, 'idrid': 100}       # DRG-Net published: 1500 (FGADR) / 2500 (IDRiD)
SEG_EVAL_EVERY = {'fgadr': 20, 'idrid': 20}     # DRG-Net: 20 / 40. 20 for IDRiD so a 100-epoch run
                                                # is evaluated (and best-saved) 5 times, not 2
SEG_SAVE_EVERY = 5                              # resume checkpoint (model_last.pth.tar) interval
CLS_EPOCHS = {'fgadr': None, 'idrid': None, 'eyepacs': None}  # None = DRG-Net's 20 / 60 / 25
RESUME = True     # continue every run from its last checkpoint on Drive (False = start over)

In [ ]:
# Unzip datasets locally (same conventions as notebooks 01-03 and 09).
import shutil
os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FGADR-Seg-set_Release.zip"
!unzip -q -n "$DRIVE_DATA_DIR/archive.zip" -d IDRiD_dataset_root
_c = glob.glob('/content/data/IDRiD_dataset_root/**/IDRiD_dataset', recursive=True)
if _c and not os.path.isdir('/content/data/IDRiD_dataset'):
    shutil.move(_c[0], '/content/data/IDRiD_dataset')
IDRID_ROOT = '/content/data/IDRiD_dataset' if os.path.isdir('/content/data/IDRiD_dataset') else '/content/data/IDRiD_dataset_root'
print('IDRiD root:', IDRID_ROOT, os.listdir(IDRID_ROOT))

_seg = glob.glob('/content/data/**/Seg-set', recursive=True)
assert _seg, 'No FGADR Seg-set folder found -- check FGADR-Seg-set_Release.zip'
FGADR_ROOT = '/content/data/FGADR-Seg-set_Release/Seg-set'
if _seg[0] != FGADR_ROOT and not os.path.exists(FGADR_ROOT):
    os.makedirs(os.path.dirname(FGADR_ROOT), exist_ok=True); os.symlink(_seg[0], FGADR_ROOT)

if RUN['eyepacs_cls']:
    _z = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, '*eyepacs*.zip')) + glob.glob(os.path.join(DRIVE_DATA_DIR, '*EyePACS*.zip')))
    assert _z, 'No EyePACS zip on Drive (see notebook 09)'
    EYEPACS_DIR = '/content/data/eyepacs'
    !unzip -q -n "{_z[0]}" -d "{EYEPACS_DIR}"

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}
REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"
%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch matplotlib scipy torchmetrics torch-fidelity

# DRG-Net reference implementation + the same compatibility patches as notebooks 02/03.
%cd /content
if not os.path.isdir('/content/dr-joint-learning'):
    !git clone https://github.com/DFKI-Interactive-Machine-Learning/dr-joint-learning.git dr-joint-learning
    !cd dr-joint-learning && git checkout 0da1bbe885f438390a0e94ec486283d9b21d5547
DRG = '/content/dr-joint-learning'
!sed -i 's/fillcolor=aug_args.value_fill/fill=aug_args.value_fill/' {DRG}/dr_classification/data/transforms.py
_f = f'{DRG}/dr_segmentation/transform/functional.py'
_t = open(_f).read().replace('from PIL import Image, ImageOps, ImageEnhance, PILLOW_VERSION',
                             "from PIL import Image, ImageOps, ImageEnhance\nPILLOW_VERSION = '9.0.0'")
open(_f, 'w').write(_t)
!pip install -q munch self-attention-cv pretrainedmodels cnn_finetune ipdb tensorboard
# Checkpoint/resume support (DRG-Net's scripts can't resume on their own). Idempotent.
!python {REPO_CODE_DIR}/module1/drgnet_resume_patch.py {DRG}
%cd {REPO_CODE_DIR}

## Classification (DRG-Net `dr_classification/main.py`)

Each run prints `test acc` / `quadratic kappa` twice at the end -- for
`best_validation_weights.pt` and for `final_weights.pt` (DRG-Net's own `main.py`). Logs are
tee'd to Drive and summarised at the bottom of this notebook.

In [ ]:
import re

def make_config(src_yaml, name, pkl):
    """Copy one of our DRG-Net config copies with data_index/save_path/log_path pointed at the
    replication folders (and epochs if CLS_EPOCHS overrides them). Nothing else changes."""
    s = open(os.path.join(REPO_CODE_DIR, 'module1', 'configs', src_yaml)).read()
    if CLS_EPOCHS.get(name):
        s = re.sub(r"(?m)^(\s+)epochs: \d+", rf"\g<1>epochs: {CLS_EPOCHS[name]}", s, count=1)
    s = re.sub(r"data_index: .*", f"data_index: '{pkl}'", s, count=1)
    s = re.sub(r"save_path: .*", f"save_path: '{REPL_DIR}/classification/{name}/saves'", s, count=1)
    s = re.sub(r"log_path: .*", f"log_path: '{REPL_DIR}/classification/{name}/logs'", s, count=1)
    os.makedirs(f'{REPL_DIR}/classification/{name}/saves', exist_ok=True)
    os.makedirs(f'{REPL_DIR}/classification/{name}/logs', exist_ok=True)
    dst = os.path.join(DRG, 'dr_classification', 'configs', f'{name}_drgnet_repl.yaml')
    open(dst, 'w').write(s)
    return f'configs/{name}_drgnet_repl.yaml'

FGADR_PKL = os.path.join(REPL_PKL_DIR, 'fgadr_release_split.pkl')
IDRID_PKL = os.path.join(REPL_PKL_DIR, 'idrid_official_split.pkl')
EYEPACS_PKL = os.path.join(REPL_PKL_DIR, 'eyepacs_official_split.pkl')
# DRG_CLS_RESUME=1 -> train() continues from <save_path>/resume_state.pt if it exists.
CLS_ENV = 'DRG_CLS_RESUME=1' if RESUME else 'DRG_CLS_RESUME=0'

%cd {REPO_CODE_DIR}
# FGADR: DRG-Net's released split, verified file-by-file against their fgadr_data_dict.pkl.
!python module1/dataprep/make_fgadr_classification_pkl.py --fgadr-root "{FGADR_ROOT}" --out "{FGADR_PKL}" --split release
_idrid_grading = glob.glob(f'{IDRID_ROOT}/**/B. Disease Grading', recursive=True)
assert _idrid_grading, f"No 'B. Disease Grading' under {IDRID_ROOT}"
!python module1/dataprep/make_idrid_classification_pkl.py --idrid-root "{_idrid_grading[0]}" --out "{IDRID_PKL}"
if RUN['eyepacs_cls']:
    !python module1/dataprep/make_eyepacs_classification_pkl.py --eyepacs-root "{EYEPACS_DIR}" --out "{EYEPACS_PKL}" --split official

In [ ]:
%cd {DRG}/dr_classification
if RUN['fgadr_cls']:
    _cfg = make_config('fgadr_poc.yaml', 'fgadr', FGADR_PKL)
    !{CLS_ENV} python main.py -c {_cfg} -overwrite 2>&1 | tee -a "{REPL_DIR}/logs/cls_fgadr.log"

In [ ]:
%cd {DRG}/dr_classification
if RUN['idrid_cls']:
    _cfg = make_config('idrid_poc.yaml', 'idrid', IDRID_PKL)
    !{CLS_ENV} python main.py -c {_cfg} -overwrite 2>&1 | tee -a "{REPL_DIR}/logs/cls_idrid.log"

In [ ]:
# EyePACS: ~35k training images x 25 epochs at 512 px -- several hours on a Colab GPU, and
# DRG-Net's main.py cannot resume mid-run, so use a long session (Colab Pro) for this one.
%cd {DRG}/dr_classification
if RUN['eyepacs_cls']:
    _cfg = make_config('eyepacs_poc.yaml', 'eyepacs', EYEPACS_PKL)
    !{CLS_ENV} python main.py -c {_cfg} -overwrite 2>&1 | tee -a "{REPL_DIR}/logs/cls_eyepacs.log"

## Segmentation -- FGADR (DRG-Net `dr_segmentation/train_fgadr.py`)

One U-Net (ResNet50 encoder) per lesion, `--seed 765 --preprocess 2` (contrast enhancement), as in
DRG-Net's README. Every `SEG_EVAL_EVERY` epochs it evaluates AP/AUC on the 20% eval slice and saves
`model_2.pth.tar` on a new best AP; every `SEG_SAVE_EVERY` epochs it saves `model_last.pth.tar`
for resuming. DRG-Net's `results/` folder is linked to Drive, so both land on Drive immediately.

In [ ]:
SEG_DRG = f'{DRG}/dr_segmentation'
SEG_OUT = os.path.join(REPL_DIR, 'segmentation')
# DRG-Net writes checkpoints to dr_segmentation/results/<run>/ -- point that at Drive so a
# disconnect never loses them (FGADR runs: models_FGADR_NO_TATL_<lesion>, IDRiD: new_models_<lesion>).
_res = os.path.join(SEG_DRG, 'results')
if os.path.isdir(_res) and not os.path.islink(_res):
    for _d in os.listdir(_res):
        _src = os.path.join(_res, _d)
        if os.path.isdir(_src):
            shutil.copytree(_src, os.path.join(SEG_OUT, _d), dirs_exist_ok=True)
    shutil.rmtree(_res)
if not os.path.islink(_res):
    os.symlink(SEG_OUT, _res)

def seg_env(run_name, dataset):
    """Env vars for one lesion run: epochs/eval/save intervals, plus resume state if any."""
    env = (f"DRG_SEG_EPOCHS={SEG_EPOCHS[dataset]} DRG_SEG_EVAL_EVERY={SEG_EVAL_EVERY[dataset]} "
           f"DRG_SEG_SAVE_EVERY={SEG_SAVE_EVERY}")
    last = os.path.join(SEG_OUT, run_name, 'model_last.pth.tar')
    if RESUME and os.path.isfile(last):
        import torch
        st = torch.load(last, map_location='cpu', weights_only=False)
        if st['epoch'] + 1 >= SEG_EPOCHS[dataset]:
            print(f'  {run_name}: already finished ({st["epoch"] + 1} epochs) -- skipping')
            return None
        print(f'  {run_name}: resuming at epoch {st["epoch"] + 2}/{SEG_EPOCHS[dataset]} (best AP so far {st["best_ap"]:.4f})')
        env += f' DRG_SEG_RESUME="{last}" DRG_SEG_BEST_AP={st["best_ap"]}'
    return env

if RUN['fgadr_seg']:
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'data', 'DR_Seg_Grading_Label_Filtered.csv'),
                os.path.join(FGADR_ROOT, 'DR_Seg_Grading_Label_Filtered.csv'))
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'configs', 'config_fgadr_seg_poc.py'),
                os.path.join(SEG_DRG, 'config_fgadr.py'))
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        print(f'=== FGADR {lesion}: {SEG_EPOCHS["fgadr"]} epochs ===')
        _env = seg_env(f'models_FGADR_NO_TATL_{lesion.lower()}', 'fgadr')
        if _env is None:
            continue
        !{_env} python train_fgadr.py --seed 765 --preprocess 2 --lesion {lesion} 2>&1 | grep -v "^img:\|^Mask:" | tail -5
        for _log in glob.glob(f'{SEG_DRG}/ap_during_learning_FGADR_NO_TATL_{lesion}*.txt'):
            shutil.copy(_log, os.path.join(REPL_DIR, 'logs'))

## Segmentation -- IDRiD (DRG-Net `dr_segmentation/train.py`)

`prepare_idrid_segmentation.py` lays IDRiD's "A. Segmentation" set out the way DRG-Net's
`utils.get_images()` reads it, and generates the field-of-view masks DRG-Net needs (IDRiD doesn't
ship them; checked against DRG-Net's bundled IDRiD samples). Then `train.py` per lesion (eval every
`SEG_EVAL_EVERY` epochs on the 20% eval slice, resumable like FGADR), and DRG-Net's
`evaluate_model.py` on the official 27-image test set.

In [ ]:
IDRID_SEG_DIR = '/content/data/IDRiD_seg_drgnet'
if RUN['idrid_seg']:
    %cd {REPO_CODE_DIR}
    !python module1/dataprep/prepare_idrid_segmentation.py --idrid-root "{IDRID_ROOT}" --out "{IDRID_SEG_DIR}"
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'configs', 'config_idrid_seg.py'),
                os.path.join(SEG_DRG, 'config.py'))
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        print(f'=== IDRiD {lesion}: {SEG_EPOCHS["idrid"]} epochs ===')
        _env = seg_env(f'new_models_{lesion.lower()}', 'idrid')
        if _env is None:
            continue
        !{_env} python train.py --seed 765 --preprocess 2 --lesion {lesion} 2>&1 | tail -5
        for _log in glob.glob(f'{SEG_DRG}/new_ap_during_learning_{lesion}*.txt'):
            shutil.copy(_log, os.path.join(REPL_DIR, 'logs'))

In [ ]:
# Test-set evaluation.
#  FGADR: our evaluate_segmentation_dice_iou.py -- AP / ROC-AUC on the raw soft predictions
#         (plus Dice/IoU), on the held-out last 20%. DRG-Net's own 'test' phase returns ALL 401
#         images incl. the training ones, so we use the held-out slice instead -- say so.
#  IDRiD: DRG-Net's own evaluate_model.py on the official 27-image test set. NB its printout
#         says "AUC:" but the number is average precision (its eval_model returns AP).
results = {}
if RUN['fgadr_seg']:
    %cd {REPO_CODE_DIR}/module1
    for lesion in SEG_LESIONS:
        ck = os.path.join(SEG_OUT, f'models_FGADR_NO_TATL_{lesion.lower()}', 'model_2.pth.tar')
        if os.path.isfile(ck):
            # 1280 = FGADR's native size; DRG-Net evaluates on full-resolution images.
            _csv = f'{REPO_CODE_DIR}/module1/data/DR_Seg_Grading_Label_Filtered.csv'
            _o = !python evaluate_segmentation_dice_iou.py --fgadr-root "{FGADR_ROOT}" --filtered-csv "{_csv}" --checkpoint "{ck}" --lesion {lesion} --image-size 1280
            print('\n'.join(_o[-5:]))
            _m = {k: float(re.search(rf'{k}:\s+([0-9.]+)', '\n'.join(_o)).group(1))
                  for k in ('Mean Dice', 'Mean IoU', 'Mean AUC-PR', 'Mean AUC-ROC')
                  if re.search(rf'{k}:\s+([0-9.]+)', '\n'.join(_o))}
            results[f'fgadr_seg_{lesion}'] = _m
if RUN['idrid_seg']:
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        ck = os.path.join(SEG_OUT, f'new_models_{lesion.lower()}', 'model_2.pth.tar')
        if os.path.isfile(ck):
            _o = !python evaluate_model.py --seed 765 --preprocess 2 --lesion {lesion} --model "{ck}"
            _ap = [l for l in _o if l.startswith('AUC:')]
            results[f'idrid_seg_{lesion}_AP'] = float(_ap[-1].split()[-1]) if _ap else None
            print(lesion, _ap[-1] if _ap else _o[-5:])

In [ ]:
# Summary table: classification acc/kappa (best-validation and final checkpoints) from the
# tee'd logs, segmentation AP/AUC from the eval outputs above. Saved to Drive as JSON.
def parse_cls(log):
    if not os.path.isfile(log):
        return None
    t = open(log).read()
    # last two = this run's best-validation and final checkpoints (logs are appended across resumes)
    accs = re.findall(r'test acc: ([0-9.]+)', t)[-2:]
    kappas = re.findall(r'quadratic kappa: (-?[0-9.]+)', t)[-2:]
    return {'best_validation': {'acc': accs[0] if accs else None, 'kappa': kappas[0] if kappas else None},
            'final': {'acc': accs[1] if len(accs) > 1 else None, 'kappa': kappas[1] if len(kappas) > 1 else None}}

for name in ('fgadr', 'idrid', 'eyepacs'):
    results[f'cls_{name}'] = parse_cls(os.path.join(REPL_DIR, 'logs', f'cls_{name}.log'))
json.dump(results, open(os.path.join(REPL_DIR, 'replication_results.json'), 'w'), indent=2)
print(json.dumps(results, indent=2))

## Using these checkpoints in the pipeline

The rest of the pipeline (notebooks 04_05-10) reads the POC checkpoints from
`module1_runs/classification/fgadr/saves` and `module1_runs/segmentation`. To use these
replication models instead, point those notebooks' `CLS_SAVES` / `SEG_DIR` at
`module1_runs/drgnet_replication/classification/fgadr/saves` and
`module1_runs/drgnet_replication/segmentation`. An EyePACS-trained classifier also needs
`--classifier-norm eyepacs` (its normalisation differs from FGADR's).